# 06 — Burn trajectory synthesis (Tier B)

This is the biggest piece of Phase 2 and unlocks Experiments 3, 5 and 6.
The monthly spending history between funding rounds doesn't exist in the
data -- this notebook manufactures it, honestly, from what does exist.

**Approach:**
1. Recover *implied* monthly burn wherever two consecutive funding events
   bracket an interval, or a funding event is followed by an observed
   shutdown (a known amount of capital consumed over a known duration).
2. Fit a regression of `log(implied burn)` on funding magnitude, months
   elapsed, and sector -- this is the "population-fitted relationship"
   the IDF describes.
3. Generate a constrained monthly trajectory per interval: interpolate
   using the fitted expectation, then **rescale** the path so it exactly
   reproduces the known start/end capital constraint. This satisfies the
   IDF's hard constraint (reproduce every observed funding event, and
   reach zero at an observed shutdown) deterministically, rather than by
   rejection-sampling a stochastic walk -- simpler, and equally correct
   for the constraint itself.
4. Validate the *fitted regression* (not the rescaling, which is exact by
   construction) via held-out prediction -- Experiment 3.

Reads: `data/processed/outcomes.csv`, `data/raw/funding_rounds.csv`
Writes: `data/processed/synthetic_trajectories.csv`, `reports/experiment3_synthesis_fidelity.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

RAW = "../data/raw"
PROCESSED = "../data/processed"
REPORTS = "../reports"

outcomes = pd.read_csv(f"{PROCESSED}/outcomes.csv")
fr = pd.read_csv(f"{RAW}/funding_rounds.csv", encoding="ISO-8859-1", low_memory=False)
fr["funded_at"] = pd.to_datetime(fr["funded_at"], errors="coerce")
fr = fr.dropna(subset=["funded_at"])
fr = fr[fr["raised_amount_usd"].fillna(0) > 0]
print(f"[load] {len(outcomes):,} companies, {len(fr):,} funding rounds")

[load] 98,280 companies, 46,817 funding rounds


## Step 1 -- recover implied burn from bracketing events

In [2]:
SNAPSHOT_DATE = pd.Timestamp("2013-12-12")

fr_sorted = fr.sort_values(["object_id", "funded_at"]).copy()
outcome_lookup = outcomes.set_index("id")[["category_code", "duration_months", "event", "founded_at"]] \
    if "founded_at" in outcomes.columns else outcomes.set_index("id")[["category_code", "duration_months", "event"]]
sector_lookup = outcomes.set_index("id")["category_code"].to_dict()
event_lookup = outcomes.set_index("id")["event"].to_dict()

intervals = []
for obj_id, g in fr_sorted.groupby("object_id"):
    g = g.reset_index(drop=True)
    for i in range(len(g) - 1):
        raised = g.loc[i, "raised_amount_usd"]
        t0, t1 = g.loc[i, "funded_at"], g.loc[i + 1, "funded_at"]
        months = (t1 - t0).days / 30.44
        if months < 3:  # sub-3-month gaps are tranches of one round, not funding cycles
            continue
        intervals.append({
            "object_id": obj_id, "raised_at_start": raised, "months_elapsed": months,
            "round_number": i, "implied_burn": raised / months,
            "sector": sector_lookup.get(obj_id, "other"), "is_terminal": False,
        })
    # terminal interval: last round -> shutdown, if this company actually closed
    last = g.iloc[-1]
    if event_lookup.get(obj_id) == 1:
        # approximate remaining cash as the last round's raised amount (a stated,
        # documented simplification -- see note below)
        dur = outcome_lookup.loc[obj_id, "duration_months"] if obj_id in outcome_lookup.index else np.nan
        if pd.notna(dur) and dur > 0:
            intervals.append({
                "object_id": obj_id, "raised_at_start": last["raised_amount_usd"],
                "months_elapsed": max(dur, 1), "round_number": len(g) - 1,
                "implied_burn": last["raised_amount_usd"] / max(dur, 1),
                "sector": sector_lookup.get(obj_id, "other"), "is_terminal": True,
            })

intervals = pd.DataFrame(intervals)
print(f"[recovered] {len(intervals):,} implied-burn intervals from {intervals['object_id'].nunique():,} companies")
print(f"[recovered] median implied monthly burn: ${intervals['implied_burn'].median():,.0f}")
print()
print("NOTE: the terminal-interval (last-round-to-shutdown) burn approximates remaining")
print("cash as the last round's raised amount. This is a stated simplification -- it")
print("ignores revenue and any unspent runway carried forward -- and should be labelled")
print("as such in the paper, consistent with the IDF's own caution on this assumption.")

[recovered] 24,184 implied-burn intervals from 15,563 companies
[recovered] median implied monthly burn: $126,833

NOTE: the terminal-interval (last-round-to-shutdown) burn approximates remaining
cash as the last round's raised amount. This is a stated simplification -- it
ignores revenue and any unspent runway carried forward -- and should be labelled
as such in the paper, consistent with the IDF's own caution on this assumption.


## Step 2 -- fit the conditional spend-rate model

In [3]:
feat = intervals.copy()
feat["log_burn"] = np.log1p(feat["implied_burn"])
feat["log_raised"] = np.log1p(feat["raised_at_start"])

top_sectors = feat["sector"].value_counts().head(8).index
feat["sector_grp"] = feat["sector"].where(feat["sector"].isin(top_sectors), "other")
feat = pd.get_dummies(feat, columns=["sector_grp"], drop_first=True)

# NOTE: months_elapsed is deliberately EXCLUDED from the features. implied_burn is
# defined as raised / months_elapsed, so including months_elapsed as a predictor of
# log_burn would let the model reconstruct its own definition (R^2 > 0.9 for free)
# rather than learn a genuine cross-venture spending pattern. round_number (which
# funding round this is for the company) is used instead as a real, non-tautological
# covariate.
feature_cols = ["log_raised", "round_number"] + [c for c in feat.columns if c.startswith("sector_grp_")]
X = feat[feature_cols].fillna(0)
y = feat["log_burn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)
print(f"[fit] R^2 on train: {train_r2:.3f}, on held-out test: {test_r2:.3f}")

resid_std = (y_train - model.predict(X_train)).std()
print(f"[fit] residual std (log space): {resid_std:.3f}  <- used to add realistic noise when generating")

[fit] R^2 on train: 0.784, on held-out test: 0.790
[fit] residual std (log space): 0.934  <- used to add realistic noise when generating


## Experiment 3 -- fidelity of the fitted model (held-out prediction)

This is the actual validation the IDF calls for: predict burn for intervals
the model never saw, using only the information available at the start of
that interval, and compare against the real implied burn.

In [4]:
pred_log = model.predict(X_test)
pred_burn = np.expm1(pred_log)
actual_burn = np.expm1(y_test)

mape = mean_absolute_percentage_error(actual_burn, pred_burn) * 100
rmse = np.sqrt(mean_squared_error(actual_burn, pred_burn))
ape = (np.abs(actual_burn - pred_burn) / actual_burn) * 100
median_ape = np.median(ape)

print(f"[Experiment 3] MAPE on held-out intervals:        {mape:.1f}%")
print(f"[Experiment 3] median APE on held-out intervals:  {median_ape:.1f}%   <- more robust to the skew below")
print(f"[Experiment 3] RMSE on held-out intervals:        ${rmse:,.0f}")
print()
print(f"[Experiment 3] R^2 in log space was {test_r2:.3f} -- these look inconsistent with a high MAPE,")
print(f"[Experiment 3] but they are not. Burn amounts span orders of magnitude, so a modest log-space")
print(f"[Experiment 3] error (residual std {resid_std:.2f}, i.e. typical predictions off by ~{np.exp(resid_std):.1f}x)")
print(f"[Experiment 3] becomes a huge PERCENTAGE error for the smallest-burn companies. Report both")
print(f"[Experiment 3] the log-space R^2 and this MAPE/median-APE, with this explanation -- do not")
print(f"[Experiment 3] drop either number, and do not report MAPE alone without the R^2 context.")

# disaggregate by sector, as the IDF's Experiment 3 asks for
disagg = pd.DataFrame({"actual": actual_burn, "predicted": pred_burn, "sector": feat.loc[X_test.index, "sector"]})
by_sector = disagg.groupby("sector").apply(
    lambda d: pd.Series({
        "n": len(d),
        "mape_pct": mean_absolute_percentage_error(d["actual"], d["predicted"]) * 100 if len(d) > 3 else np.nan,
    })
).sort_values("n", ascending=False)
print("\n[Experiment 3] MAPE by sector:")
print(by_sector.head(10))

by_sector.to_csv("../reports/experiment3_synthesis_fidelity.csv")
print("\n[saved] ../reports/experiment3_synthesis_fidelity.csv")

[Experiment 3] MAPE on held-out intervals:        108.2%
[Experiment 3] median APE on held-out intervals:  55.8%   <- more robust to the skew below
[Experiment 3] RMSE on held-out intervals:        $2,463,225

[Experiment 3] R^2 in log space was 0.790 -- these look inconsistent with a high MAPE,
[Experiment 3] but they are not. Burn amounts span orders of magnitude, so a modest log-space
[Experiment 3] error (residual std 0.93, i.e. typical predictions off by ~2.5x)
[Experiment 3] becomes a huge PERCENTAGE error for the smallest-burn companies. Report both
[Experiment 3] the log-space R^2 and this MAPE/median-APE, with this explanation -- do not
[Experiment 3] drop either number, and do not report MAPE alone without the R^2 context.

[Experiment 3] MAPE by sector:
                 n    mape_pct
sector                        
software     900.0  109.355691
biotech      748.0  114.716931
web          492.0   89.074991
enterprise   417.0   93.357107
mobile       370.0  109.538815
advertis

C:\Users\ranga\AppData\Local\Temp\ipykernel_612\2912257211.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_sector = disagg.groupby("sector").apply(


## Step 3 -- generate constrained monthly trajectories

In [5]:
def generate_trajectory(row, model, feature_cols, resid_std, rng):
    """One synthesized monthly trajectory for a single interval, constrained
    to consume exactly the known capital over the known duration."""
    n_months = max(1, int(round(row["months_elapsed"])))
    x = pd.DataFrame([{c: row.get(c, 0) for c in feature_cols}])
    expected_log_burn = model.predict(x)[0]
    monthly = np.expm1(expected_log_burn + rng.normal(0, resid_std, size=n_months))
    monthly = np.clip(monthly, 1, None)
    # hard constraint: rescale so the path consumes exactly the known capital
    scale = row["raised_at_start"] / monthly.sum() if monthly.sum() > 0 else 1.0
    return monthly * scale

rng = np.random.default_rng(42)
SAMPLE_CAP = 15000  # raised from the original 2000 -- that cap was throwing away >90%
# of your real 24,184 intervals, which is why only ~1,900 companies reached fusion
# in notebook 09. Set to None to use every interval if runtime allows.
sample = feat if SAMPLE_CAP is None else feat.sample(n=min(SAMPLE_CAP, len(feat)), random_state=42)

records = []
for _, row in sample.iterrows():
    traj = generate_trajectory(row, model, feature_cols, resid_std, rng)
    for month_idx, spend in enumerate(traj):
        records.append({
            "object_id": row["object_id"], "month_idx": month_idx, "synthesized_spend": spend,
            "provenance": "synthesized", "is_terminal_interval": row["is_terminal"],
        })

synthetic = pd.DataFrame(records)
synthetic.to_csv("../data/processed/synthetic_trajectories.csv", index=False)
print(f"[done] generated {len(synthetic):,} monthly synthetic spend values across {sample['object_id'].nunique():,} companies")
print(f"[done] wrote ../data/processed/synthetic_trajectories.csv")
print()
print(f"Sample size is capped at {SAMPLE_CAP} intervals -- set SAMPLE_CAP = None above for the full population.")

[done] generated 537,251 monthly synthetic spend values across 11,050 companies
[done] wrote ../data/processed/synthetic_trajectories.csv

Sample size is capped at 15000 intervals -- set SAMPLE_CAP = None above for the full population.
